# Mestres do XGH — Aula 8
## Arquivos: o dado que sobrevive

**Shift + Enter** roda a célula.

Hoje o Python sai do navegador e entra na **sua máquina**.

---
# 1. A pergunta que ficou de sábado passado

Você fechou o Colab e a agenda de contatos sumiu. A turma sumiu. Tudo sumiu.

Porque tudo que a gente fez em sete aulas viveu na **memória** — e memória morre quando o
programa fecha.

A saída é o **arquivo**: um pedaço de dado que fica no disco depois que tudo acaba.

---
# 2. Escrever num arquivo

In [ ]:
with open("notas.txt", "w") as f:
    f.write("Ana 8.5\n")
    f.write("Bruno 4.0\n")
    f.write("Carla 9.5\n")

print("pronto")

Três coisas nessa linha, e vale conhecer as três:

```
with open("notas.txt", "w") as f:
 │     │        │        │     └── o apelido do arquivo aberto
 │     │        │        └──────── o MODO: "w" = write (escrever)
 │     │        └───────────────── o nome do arquivo
 │     └────────────────────────── abre
 └──────────────────────────────── e FECHA sozinho no fim do bloco
```

O `with` é o detalhe que importa. Sem ele você teria que lembrar de fechar o arquivo, e
esquecer disso é uma das causas clássicas de dado corrompido. **Com `with`, o Python fecha por
você — inclusive se der erro no meio.**

Repare também no `\n`: é a quebra de linha. Sem ele, tudo sai grudado.

---
# 3. Ler

In [ ]:
with open("notas.txt") as f:
    conteudo = f.read()

print(conteudo)

Sem o segundo argumento, o modo é `"r"` (*read*) — é o padrão.

E como quase sempre você quer linha por linha:

In [ ]:
with open("notas.txt") as f:
    for linha in f:
        print(repr(linha))

```
'Ana 8.5\n'
'Bruno 4.0\n'
'Carla 9.5\n'
```

Usei `repr()` de propósito para o `\n` aparecer. **Toda linha vem com a quebra grudada no fim** —
e é aí que entra o `.strip()` da Aula 5:

In [ ]:
with open("notas.txt") as f:
    for linha in f:
        nome, nota = linha.strip().split()
        print(f"{nome:6s} {float(nota)}")

Repare no `float(nota)`. **Tudo que sai de um arquivo é texto** — exatamente como o `input()`
da Aula 2. O arquivo não sabe o que é número; quem sabe é você.

---
# 4. Os três erros que apagam o seu trabalho

### 💥 Quebrando de propósito nº 1 — o modo `"w"` é destrutivo

O arquivo tem três nomes. Vou acrescentar um quarto:

In [ ]:
with open("notas.txt", "w") as f:
    f.write("Diego 6.0\n")

with open("notas.txt") as f:
    print(f.read())

```
Diego 6.0
```

**Os três sumiram.** Sem aviso, sem erro, sem confirmação.

`"w"` não quer dizer "escrever no fim". Quer dizer **"zerar o arquivo e escrever"**. Se o arquivo
existia, o conteúdo antigo já era.

Para acrescentar, o modo é `"a"` (*append* — o mesmo nome do método de lista da Aula 5):

In [ ]:
with open("notas.txt", "a") as f:
    f.write("Elis 7.5\n")

with open("notas.txt") as f:
    print(f.read())

| Modo | O que faz | Se o arquivo existe |
|---|---|---|
| `"r"` | lê | — |
| `"w"` | escreve | **apaga tudo antes** |
| `"a"` | acrescenta no fim | preserva |

> Guarde: **`"w"` é a faca do dia.** Quase todo iniciante perde trabalho com ela pelo menos uma
> vez. Na dúvida, use `"a"`.

### 💥 Quebrando de propósito nº 2 — o arquivo que não existe

In [ ]:
with open("nao_existe.txt") as f:
    print(f.read())

```
FileNotFoundError: [Errno 2] No such file or directory: 'nao_existe.txt'
```

Esse erro é **inevitável** no mundo real: o arquivo pode ter sido movido, renomeado, apagado, ou
o programa está rodando de outra pasta.

E aqui a gente esbarra num problema que vem se acumulando desde a **Aula 2**: como é que o
programa continua vivo quando algo dá errado?

*(Semana que vem. É a Aula 9, e ela finalmente paga essa dívida.)*

### 💥 Quebrando de propósito nº 3 — esquecer o `\n`

In [ ]:
with open("colado.txt", "w") as f:
    for nome in ["Ana", "Bruno", "Carla"]:
        f.write(nome)

with open("colado.txt") as f:
    print(repr(f.read()))

```
'AnaBrunoCarla'
```

Nenhum erro. Só um arquivo inútil.

---
# 5. JSON em arquivo — a agenda que finalmente sobrevive

Aqui as duas últimas aulas se encontram. A agenda da semana passada, agora com endereço fixo:

In [ ]:
import json

agenda = [
    {"nome": "Ana", "telefone": "11 90000-0001", "cidade": "Santos"},
    {"nome": "Bruno", "telefone": "13 90000-0002", "cidade": "Guarujá"},
]

with open("agenda.json", "w", encoding="utf-8") as f:
    json.dump(agenda, f, indent=2, ensure_ascii=False)

print(open("agenda.json", encoding="utf-8").read())

In [ ]:
with open("agenda.json", encoding="utf-8") as f:
    de_volta = json.load(f)

print(type(de_volta))
print(de_volta[0]["nome"])

> **Atenção ao nome:** é `json.dump` e `json.load`, **sem o `s`**.
>
> - `dumps` / `loads` → mexem com **string** (o `s` é de string)
> - `dump` / `load` → mexem com **arquivo**
>
> Trocar um pelo outro é erro de todo dia. Quando der erro estranho em JSON, cheque o `s` antes
> de qualquer coisa.

---
# 6. CSV — a planilha do mundo real

CSV é texto separado por vírgula. Excel abre, Google Sheets abre, e é o formato em que dado
tabular circula no mundo inteiro.

In [ ]:
import csv

with open("turma.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["nome", "nota", "faltas"])
    escritor.writerow(["Ana", 8.5, 2])
    escritor.writerow(["Bruno", 4.0, 7])
    escritor.writerow(["Carla", 9.5, 0])

print(open("turma.csv", encoding="utf-8").read())

E para ler, o `DictReader` transforma **cada linha num dicionário**, usando o cabeçalho como
chaves. É a Aula 7 caindo do céu:

In [ ]:
with open("turma.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f):
        print(linha)

Olhe com atenção:

```
{'nome': 'Ana', 'nota': '8.5', 'faltas': '2'}
```

O `8.5` está **entre aspas**. É texto.

> **Tudo que sai de arquivo é texto.** Mesmo que pareça número. É o mesmo problema do `"2" + "2"`
> que abriu a Aula 2 — ele volta em toda aula em que o dado vem de fora.

In [ ]:
soma = 0
with open("turma.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f):
        soma += float(linha["nota"])      # o float() é obrigatório

print(f"média: {soma / 3:.2f}")

### 🔧 Exercício

1. Salve a `turma` da Aula 7 num arquivo `turma.json`
2. Leia de volta e imprima só os aprovados
3. Acrescente um aluno e salve de novo — **sem perder os que já estavam lá**

In [ ]:
# sua vez

---
# 💊 Pílula de IA — sem dado guardado não existe modelo

## Etapa 1 — o dataset é um arquivo

Todo modelo que vocês viram neste curso saiu de dados que alguém digitou. O detector de spam da
Aula 3 nasceu de 16 mensagens escritas na mão, **dentro do código**.

Isso não é como funciona de verdade. Na vida real o dado mora num arquivo. Vamos fazer certo:

In [ ]:
import csv

mensagens = [
    ("promocao imperdivel clique aqui agora", "spam"),
    ("voce ganhou um premio gratis resgate ja", "spam"),
    ("credito pre aprovado clique no link", "spam"),
    ("clique aqui e ganhe dinheiro rapido", "spam"),
    ("oi tudo bem vamos almocar amanha", "normal"),
    ("reuniao adiada para as 15h", "normal"),
    ("segue em anexo o relatorio do mes", "normal"),
    ("a aula de sabado comeca as 10h", "normal"),
]

with open("spam.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["texto", "rotulo"])
    for texto, rotulo in mensagens:
        escritor.writerow([texto, rotulo])

print(open("spam.csv", encoding="utf-8").read())

Agora o modelo é treinado **a partir do arquivo** — o código não conhece nenhuma mensagem:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

textos, rotulos = [], []
with open("spam.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f):
        textos.append(linha["texto"])
        rotulos.append(linha["rotulo"])

vetor = CountVectorizer()
modelo = MultinomialNB().fit(vetor.fit_transform(textos), rotulos)

print(modelo.predict(vetor.transform(["clique aqui e ganhe premio"])))
print(modelo.predict(vetor.transform(["vamos almocar depois da reuniao"])))

Repare no que mudou de verdade: **para melhorar esse modelo você edita um arquivo, não um
programa.**

É a promessa da Aula 3 virando concreta. Lá eu disse "você corrige o modelo dando mais
exemplo". Agora dá para ver onde os exemplos moram: numa planilha que qualquer pessoa abre —
inclusive quem não sabe programar.

## Etapa 2 — quanto dado é preciso?

Vamos medir. Um conjunto maior, um teste separado de 50 mensagens que o modelo nunca viu, e a
média de 30 sorteios diferentes para não depender da sorte:

In [ ]:
import random, itertools

spam_a = ["promocao imperdivel", "ultima chance", "oferta relampago", "so hoje", "aproveite agora", "chance unica"]
spam_b = ["clique aqui", "clique no link", "acesse o link", "clique no botao", "entre no site"]
spam_c = ["e ganhe dinheiro", "e resgate seu premio", "credito aprovado", "ganhe premio gratis", "dinheiro rapido"]

norm_a = ["bom dia", "oi tudo bem", "fala", "boa tarde", "opa"]
norm_b = ["consegue revisar o relatorio", "vamos almocar amanha", "a reuniao mudou de horario",
          "segue o arquivo em anexo", "esqueci a chave na mesa", "parabens pelo trabalho",
          "a aula comeca as dez", "vou chegar atrasado", "o documento esta pronto", "preciso da sua ajuda"]
norm_c = ["por favor", "quando puder", "obrigado", "abraco", "valeu"]

spam = [f"{a} {b} {c}" for a, b, c in itertools.product(spam_a, spam_b, spam_c)]
normal = [f"{a} {b} {c}" for a, b, c in itertools.product(norm_a, norm_b, norm_c)]

random.seed(1)
random.shuffle(spam)
random.shuffle(normal)

teste = [(m, "spam") for m in spam[:25]] + [(m, "normal") for m in normal[:25]]
pool = [(m, "spam") for m in spam[25:]] + [(m, "normal") for m in normal[25:]]

print(f"{len(pool)} mensagens para treinar, {len(teste)} para testar")

In [ ]:
def avaliar(n, ruido=0, repeticoes=30):
    total = 0
    rng = random.Random(0)
    for _ in range(repeticoes):
        amostra = rng.sample(pool, n)
        X = [a[0] for a in amostra]
        y = [a[1] for a in amostra]

        if ruido:
            for i in rng.sample(range(n), min(ruido, n)):
                y[i] = "normal" if y[i] == "spam" else "spam"

        v = CountVectorizer()
        m = MultinomialNB().fit(v.fit_transform(X), y)
        pred = m.predict(v.transform([t[0] for t in teste]))
        total += sum(1 for p, t in zip(pred, teste) if p == t[1])

    return total / repeticoes / len(teste) * 100


for n in [4, 8, 16, 32, 64, 128]:
    print(f"{n:4d} exemplos -> {avaliar(n):5.1f}% de acerto")

```
   4 exemplos ->  77.8% de acerto
   8 exemplos ->  94.6% de acerto
  16 exemplos ->  99.9% de acerto
  32 exemplos -> 100.0% de acerto
  64 exemplos -> 100.0% de acerto
 128 exemplos -> 100.0% de acerto
```

Duas leituras, e as duas importam:

1. **No começo, cada exemplo vale ouro.** De 4 para 16 o acerto pula de 78% para 100%.
2. **Depois, satura.** De 32 para 128 não muda nada — você quadruplicou o dado e ganhou zero.

*(Este conjunto é fácil de propósito, montado por combinação de pedaços — por isso chega a 100%.
Num problema real o número seria bem menor. O que importa aqui é o **formato da curva**, e ele é
o mesmo: sobe rápido, depois empaca.)*

Isso explica uma coisa que você vai ouvir muito: *"a gente precisa de mais dados"*. Às vezes é
verdade. Às vezes o modelo já saturou e o problema é outro.

## Etapa 3 — 💥 e se o dado estiver errado?

Agora o lado feio. Mesmos 64 exemplos, mas com alguns rótulos **trocados de propósito** — spam
marcado como normal e vice-versa. É o que acontece quando quem rotulou estava cansado:

In [ ]:
for r in [0, 4, 8, 16, 24]:
    print(f"{r:2d} rótulos errados de 64 -> {avaliar(64, ruido=r):5.1f}%")

```
 0 rótulos errados de 64 -> 100.0%
 4 rótulos errados de 64 -> 100.0%
 8 rótulos errados de 64 -> 100.0%
16 rótulos errados de 64 ->  97.7%
24 rótulos errados de 64 ->  85.9%
```

O modelo **aguenta** um pouco de erro — 8 rótulos trocados e ele nem sente. Depois de certo
ponto, desaba.

E preste atenção no que ele **não** faz: não avisa. Não dá erro. Não fica vermelho. Ele
simplesmente responde pior, com a mesma cara de confiança de sempre — igual à extrapolação da
Aula 6.

## Etapa 4 — por que isso muda a sua vida

1. **O ativo é o dado, não o código.** O código do detector de spam cabe em cinco linhas e está
   em qualquer tutorial. O que ninguém te dá é o arquivo de exemplos.
2. **Por isso empresas de IA brigam por dado.** Não é o algoritmo que é segredo.
3. **Dado ruim envenena em silêncio.** Um modelo treinado em dado errado não reclama — ele erra
   educadamente.

---

### O que fica da pílula de hoje

1. **Sem dado guardado não existe modelo.** E dado guardado é arquivo.
2. **Mais dado ajuda muito no começo e cada vez menos depois.**
3. **Dado errado é pior que dado faltando** — porque não avisa.

E é por isso que a Aula 10 é sobre **limpar tabela**: 80% do trabalho de IA acontece antes do
modelo, no arquivo.

---
# 🏠 Tarefa da semana

**Toda a tarefa desta semana é para rodar na sua máquina, no VS Code — não no Colab.**
Se a instalação não terminou em aula, o guia está no material.

### 1. Agenda que sobrevive
Programa que carrega `agenda.json` se existir, deixa você acrescentar um contato, e salva de
volta. Feche o programa, abra de novo e confirme que o contato continua lá.

### 2. Diário de bordo
Um programa que pergunta "o que você aprendeu hoje?" e **acrescenta** a resposta num
`diario.txt`, com a data. Use o modo `"a"`.

### 3. Boletim em CSV
Salve a turma num `turma.csv` e escreva um programa separado que lê o arquivo e imprime a média
e os aprovados. **Dois programas, um arquivo** — foi exatamente isso que o modelo de spam fez.

### 4. Desafio
Faça o programa da tarefa 1 funcionar mesmo quando o `agenda.json` **não existe** (primeira
execução).

Você vai ter que checar antes de abrir. Existe um jeito melhor de fazer isso, e ele é a próxima
aula.

---

### 🤔 Para pensar até sábado

No desafio acima você provavelmente escreveu algo como *"se o arquivo existir, abre"*.

Mas e se o arquivo existir e estiver corrompido? E se o usuário digitar `abc` onde você espera
número — aquela pergunta da **Aula 2** que eu ainda não respondi direito? E se o CSV tiver uma
linha faltando coluna?

São todos o mesmo problema: **o programa morre no primeiro imprevisto.**

*(A resposta é a Aula 9. E é lá que eu finalmente pago a dívida do `abc`, que está aberta desde
a segunda aula.)*